🌐 **English** | [한국어](dx_app.ko.ipynb) | [日本語](dx_app.ja.ipynb) | [中文](dx_app.zh.ipynb)

# DEEPX Tutorial 02 - DX-APP Usage


In this second tutorial, we will introduce DX-APP and learn how to run a compiled DXNN model in an AI application.

## Learning objectives

By the end of this tutorial, you will be able to:

- know that DX-APP ships ready-to-run C++ and Python example code for **every model in the [DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/)** (497 model variants), and use that code to run any model you choose;
- describe how the examples are organized by task, model family, and variant, and how the variant name ties together the model file, the executables, and the scripts;
- download only the models and videos a tutorial needs;
- run classification, detection, pose, and segmentation examples and view their saved results in the notebook;
- choose between synchronous, asynchronous, and PPU execution, and measure which Python variant suits your model;
- query the examples with `dx_tool.sh` and start the demo launcher;
- export a standalone C++ or Python example, read its model card, and run it outside the DX-APP tree;
- chain several models into one pipeline with the multi-model graph, by editing a JSON file instead of code.

## What is DX-APP?

**DX-APP** is the reference application for DEEPX NPUs. Its defining feature is coverage: **every model published in the [DEEPX Model Zoo](https://developer.deepx.ai/modelzoo/) has a matching, ready-to-run example** in both C++ and Python, organized by task (object detection, classification, face detection, pose, segmentation, depth, and more). You pick a model on the Model Zoo page, and DX-APP already contains the code that runs it. DX-APP helps developers quickly set up the runtime environment
and serves as a template for building and customizing their own AI applications.

This tutorial uses **DX-APP v3.3.0**, the version shipped with DX-All Suite v2.5.0.

For more details, find the overview from [dx_app git repo](https://github.com/DEEPX-AI/dx_app/blob/main/README.md) or download DX-APP User Guide from [here](https://developer.deepx.ai/download/?id=744)!

> **Note:** To download the User Guide, you must log in to https://developer.deepx.ai/ first.

Let's see the file structure of DX-APP:

In [ ]:
# Load the shared DEEPX tutorial paths (identical first code cell in every notebook).
from pathlib import Path; import importlib, sys
sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial_paths.py").is_file())))
import tutorial_paths; importlib.reload(tutorial_paths)  # pick up updates without restarting the kernel
from tutorial_paths import setup_tutorial
import shlex

paths = setup_tutorial(needs=["dx_rt", "dx_app", "npu"])
paths.require(*["dx_rt", "dx_app", "npu"])   # stop here with the fix if a prerequisite is missing
%cd {DX_ALL_SUITE_DIR}


In [ ]:
!tree -L 1 dx-runtime

In [ ]:
!tree -L 1 dx-runtime/dx_app
!tree -L 1 dx-runtime/dx_app/bin
#!tree -L 2 dx-runtime/dx_app/src

**DX-APP** demos are optimized to showcase pre-compiled models on DEEPX NPUs with minimal setup.
Each demo represents a common AI task and can be executed using images, videos, or live camera
input.

<img src="assets/ai-model-type.png" style="max-width: 1000px;">

---
**DX-APP Repository Layout:**
```shell
dx_app/
├── src/
│   ├── cpp_example/                    # C++ end-to-end examples (497 model variants across 29 tasks)
│   │   ├── <task>/<family>/<variant>/  #   One folder per .dxnn file: entry points, factory, config.json, README.md
│   │   ├── multi_model_graph/          #   Multi-model graph CLI and sample graph JSON files (section 4)
│   │   └── common/                     # Shared C++ runtime layer
│   │       ├── base/                   #   Abstract interfaces (factories, pre/post-processors, visualizers, inputs)
│   │       ├── processors/             #   Shared processors
│   │       ├── runner/                 #   Task-specific sync/async runner pairs
│   │       ├── inputs/                 #   Image, video, camera, and RTSP input sources
│   │       ├── visualizers/            #   Task-specific visualizers
│   │       ├── config/                 #   ModelConfig loader
│   │       ├── graph/                  #   Multi-model graph engine
│   │       ├── registry/               #   Model registry used by the graph engine
│   │       ├── trackers/               #   IoU tracker
│   │       ├── third_party/            #   Header-only third-party libraries
│   │       └── utility/                #   Labels, preprocessing, profiling, and run utilities
│   ├── python_example/                 # Python end-to-end examples (the same 497 variants and 29 tasks)
│   │   ├── <task>/<family>/<variant>/  #   Four scripts per variant, config.json, and the model card files
│   │   └── common/                     # Shared Python runtime layer (same structure as the C++ one)
│   ├── postprocess/                    # C++ post-processing (consumed by pybind11 bindings)
│   ├── utility/                        # Shared support code used by build flow
│   └── bindings/
│       └── python/
│           ├── dx_postprocess/         # pybind11 bindings wrapping src/postprocess/
│           └── dx_graph/               # pybind11 module running multi-model graphs (./build.sh --all)
├── config/
│   ├── model_registry.json             # Model registry: one row per variant, the single source of truth
│   ├── test_models.conf                # Test model configuration
│   └── README.md                       # Config directory documentation
├── scripts/                            # dx_tool.sh, extract_model_package.sh, download_models.py, and helpers
├── tests/                              # pytest-based test suites
├── assets/                             # Links to downloaded models and videos
├── sample/                             # Bundled images and task-specific sample data
├── bin/                                # C++ executables generated by the build: <variant>_sync, <variant>_async
├── build.sh                            # Build script (default: the 26 demo models; --all: every variant)
├── setup.sh                            # Model, video, and dependency setup
├── run_demo.sh                         # Demo launcher: 26 demos, up to 6 execution modes
├── run_multi_model_demo.sh             # Multi-model graph scenarios and graph checks
├── install.sh                          # Dependency and OpenCV installer
└── docs/                               # Detailed documentation
```
---
**Task, family, and variant**

Every example lives in `<task>/<family>/<variant>/`. The **variant** is the stem of the `.dxnn` file, and it names everything that belongs to that model:

| Item | For `yolov7_640x640.dxnn` |
|---|---|
| Example folders | `src/cpp_example/object_detection/yolov7/yolov7_640x640/` and the same path under `src/python_example/` |
| C++ executables | `bin/yolov7_640x640_sync`, `bin/yolov7_640x640_async` |
| Python scripts | `yolov7_640x640_sync.py`, `yolov7_640x640_async.py`, `yolov7_640x640_sync_cpp_postprocess.py`, `yolov7_640x640_async_cpp_postprocess.py` |
| Download name | `./setup.sh --models yolov7_640x640` |

> **Coming from DX-APP v3.2.x?** Executables are now named after the variant (`bin/yolov7_async` is now `bin/yolov7_640x640_async`), the PPU models moved to `object_detection/yolo_ppu/`, and several tasks were renamed or split: `classification` is now `image_classification`, `obb_detection` is `oriented_object_detection`, `image_enhancement` is `low_light_enhancement`, `face_alignment` is `face_landmark`, and `embedding` became `face_recognition`, `image_retrieval`, `visual_place_recognition`, and `zero_shot_image_classification`.

**The 29 task directories** (the same in `cpp_example` and `python_example`):

- Detection: `object_detection`, `oriented_object_detection`, `face_detection`, `hand_detection`, `mainbody_detection`, `3d_object_detection`, `anomaly_detection`
- Segmentation: `instance_segmentation`, `semantic_segmentation`, `zero_shot_instance_segmentation`, `panoptic_driving_perception`, `image_matting`
- Pose and keypoints: `pose_estimation`, `hand_landmark`, `face_landmark`, `keypoint_detection`, `object_pose_estimation`
- Classification and attributes: `image_classification`, `zero_shot_image_classification`, `person_attribute`, `face_attribute`
- Recognition and retrieval: `face_recognition`, `person_reid`, `image_retrieval`, `visual_place_recognition`
- Image restoration and depth: `image_denoising`, `super_resolution`, `low_light_enhancement`, `depth_estimation`

DX-APP v3.3.0 registered 147 more Model Zoo models, including RT-DETR (v1, v2, and v3), YOLOv12, YOLOv13, pre-optimized YOLO26, and models for the new tasks: anomaly detection (PatchCore, EfficientAD), image matting (PP-Matting), CLIP image retrieval, person re-identification, and visual place recognition (EigenPlaces).

## Prerequisites

- Tutorial 01 completed: DX-Runtime installed with `--all` (DX-RT, DX-APP, and its `venv-dx-runtime` Python environment), NPU visible as `/dev/dxrt0`.
- **DX-RT 3.5.0 or later.** DX-APP v3.3.0 downloads its models from Model Zoo release `2_5_0`, whose `.dxnn` files use container format v9. Older runtimes refuse them (see Troubleshooting). `dxrt-cli --version` prints the installed version.
- The DX-APP sample models and videos. Tutorial 01 section 3.3 downloads them; the download cells below only fetch what is still missing (about 300 MB of models for this tutorial, plus the 1.2 GB shared sample-video archive on a fresh host).
- A graphical desktop session is optional. The cells run the examples headless (`--no-display --save`) and show the saved images in the notebook; cells that open a window are switched off by a flag such as `SHOW_WINDOW = False`.
- No `sudo`. Expect about 45 minutes. Two cells take one to two minutes each: the C++ build in section 3.3 and, if the graph executables are missing, the build in section 4.1.

The next cell locates the SDK, checks these requirements, and prints a status table. Anything marked `MISSING` comes with the step that provides it.

## 1. Prepare the models and videos


1. Move to `dx_app` directory:

In [ ]:
# Move to the configured DX-APP directory.
%cd {DX_APP_DIR}

2. Download the required models and sample videos with `setup.sh`. The script is interactive (it asks which model categories to download), so it runs in a terminal, not in a cell. Open **File > New > Terminal** and run:

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./setup.sh --no-force
```

3. The next cell prints the same two commands with your actual installation path.


In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./setup.sh --no-force")


> ![](assets/sc-setup.png)

4. If you have enough free storage for the whole Model Zoo (several tens of GB), press **Enter** to download all Model Zoo models. The screenshot comes from an earlier release; the current menu lists more models and the new task names.

5. Otherwise, download only the models this tutorial needs with the code cell below. Pass the **variant name**, the `.dxnn` file stem such as `yolov7_640x640`; the older Model Zoo names such as `YoloV7` are still accepted. Tutorial 01 section 3.3 already downloaded these models; if you completed it, `--no-force` makes the cell skip every file and finish in a few seconds. Sections 3 and 4 fetch their extra models the same way.

In [ ]:
# The models used in section 2, by variant name (the .dxnn file stem).
# --no-force skips files that already exist, so rerunning this cell is cheap.
SELECTED_MODELS = shlex.join([
    "resnet50_224x224",
    "yolov7_640x640",
    "yolov7_640x640_ppu",
    "yolov8-s-pose_640x640",
    "yolov8-n-seg_640x640",
])

# Preview what would be downloaded, without downloading:
#!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --dry-run

# Download the selected models only when they are missing
!cd "{DX_APP_DIR}" && bash setup.sh --models {SELECTED_MODELS} --no-force

6. Verify that both models and videos are downloaded as expected:

In [ ]:
# Downloaded models and videos are stored under `assets` path as a symbolic link
!cd "{DX_APP_DIR}" && tree assets

In [ ]:
# AI models converted to DXNN format

# Number of downloaded models
!cd "{DX_APP_DIR}" && ls assets/models | wc -l

# Downloaded model list
!cd "{DX_APP_DIR}" && tree assets/models

In [ ]:
# video files for demo inputs
!cd "{DX_APP_DIR}" && tree assets/videos

## 2. Run Demos

**DX-APP** is a set of ready-made demo applications that show how to run compiled models on DEEPX NPUs, including classification, detection, segmentation, pose estimation, and many others.

Users can learn and build applications by exploring the diverse DX-APP demos.

The C++ executables in `bin/` are named after the variant: `bin/<variant>_sync` and `bin/<variant>_async`. They all take the same input options (`-i` image, `-v` video, `-c` camera, `-r` RTSP stream) and the same output options:

| Option | Effect |
|---|---|
| `--no-display` | Do not open a window; print the performance summary only |
| `-s`, `--save` | Save the rendered result |
| `--save-dir <dir>` | Where to save it; every run writes a new sub-directory |
| `-l`, `--loop <n>` | Repeat the input `n` times |

The cells in this section run headless with `--no-display --save` and show the saved image below the cell, so they also work over SSH and never block the notebook. The commented-out lines run the same example on a video or a camera; they open a window, so use them on a desktop session.

The next cell creates the output directory and a small helper that shows the newest saved image.

In [ ]:
import time
from IPython.display import Image, Markdown, display

DEMO_OUTPUT_DIR = WORKSPACE_DIR / "dx-app-outputs"   # results saved with --save in section 2
DEMO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Remember when each cell starts, so the helper shows only images saved by the current cell.
CELL_START = {"time": time.time()}
def _mark_cell_start(info):
    CELL_START["time"] = time.time()
ip = get_ipython()
for callback in list(ip.events.callbacks["pre_run_cell"]):
    if getattr(callback, "__name__", "") == "_mark_cell_start":
        ip.events.unregister("pre_run_cell", callback)
ip.events.register("pre_run_cell", _mark_cell_start)


def show_latest_image(folder, width=640):
    # Show the newest .jpg/.png that the current cell saved under `folder`
    # (every run saves into its own sub-directory).
    images = [p for p in Path(folder).rglob("*")
              if p.suffix.lower() in (".jpg", ".jpeg", ".png") and p.stat().st_mtime >= CELL_START["time"]]
    if not images:
        print(f"This cell saved no image under {folder}; check the command output above.")
        return
    latest = max(images, key=lambda p: p.stat().st_mtime)
    print("Saved result:", latest.relative_to(folder))
    display(Image(filename=str(latest), width=width))

### 2.1 Classification
- **bin/resnet50_224x224_sync** runs a pre-trained ImageNet classification model and draws the top predicted class on the image.

`-m` is optional for every C++ example: without it, the example loads its default model `assets/models/<variant>.dxnn`. The cells pass it anyway, so you can see where the model comes from.

In [ ]:
%cd {DX_APP_DIR}

# Help: the options shared by every C++ example
!./bin/resnet50_224x224_sync -h

# Image input, headless: classify one ImageNet sample and save the annotated image
!./bin/resnet50_224x224_sync -m assets/models/resnet50_224x224.dxnn -i sample/ILSVRC2012/1.jpeg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

### 2.2 Object Detection
- **bin/yolov7_640x640_sync** and **bin/yolov7_640x640_async** run the YOLOv7 object detector on images, videos, or a camera. The next cell lists the other YOLO executables that the default build created.

In [ ]:
%cd {DX_APP_DIR}

# Find other YOLO-based reference executables
!ls ./bin/yolo*

#### Execution Modes

DX-APP provides different execution modes to suit different application requirements:

| Mode | Main characteristic | Recommended use |
|---|---|---|
| Synchronous | Processes each frame sequentially | Simple flow and latency inspection |
| Asynchronous | Overlaps pipeline stages | Higher throughput |
| PPU | Offloads supported post-processing work | Lower host CPU overhead |

- **Synchronous application**

<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Sync_Inference_Operation.png" style="max-width: 300px;">


In [ ]:
%cd {DX_APP_DIR}

# Image input, headless: the annotated image is saved and shown below
!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -i sample/img/sample_dog.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# Video input, headless: 588 frames, prints the performance summary (about 20 seconds)
!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4 --no-display

# On a desktop session (opens a window):
#!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4
#!./bin/yolov7_640x640_sync -m assets/models/yolov7_640x640.dxnn -c 0

- **Asynchronous application**


<img src="https://github.com/DEEPX-AI/dx_rt/raw/main/docs/source/resources/09_01_Async_Inference_Operation.png" style="max-width: 1000px;">

The next cell runs the same video through the asynchronous executable. Compare its `Overall FPS` line with the synchronous run above.

In [ ]:
%cd {DX_APP_DIR}

# Video input, headless: compare "Overall FPS" with the synchronous run above
!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4 --no-display

# On a desktop session (opens a window):
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/boat.mp4
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -c 0

# RTSP input (this public stream is reachable only from Korea)
#!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -r "rtsp://210.99.70.120:1935/live/cctv002.stream"

- **PPU vs non-PPU**

The two cells below run the same 7-second clip through the non-PPU model (`bin/yolov7_640x640_async`) and the PPU model (`bin/yolov7_640x640_ppu_async`). With `SHOW_WINDOW = False` (the default) they run headless with `--no-display` and print only the FPS summary, so the notebook does not block; set `SHOW_WINDOW = True` on a desktop session to watch the detections and the FPS counter in the DX-APP window. Compare the two FPS lines: the PPU model moves the candidate filtering and class selection onto the NPU. All PPU examples are listed under `object_detection/yolo_ppu/`.

In [ ]:
SHOW_WINDOW = False   # True opens the DX-APP window (desktop session only); False prints the FPS summary and returns
%cd {DX_APP_DIR}

# PPU disabled
!./bin/yolov7_640x640_async -m assets/models/yolov7_640x640.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}

In [ ]:
SHOW_WINDOW = False   # keep the same value as in the previous cell
%cd {DX_APP_DIR}

# PPU enabled
!./bin/yolov7_640x640_ppu_async -m assets/models/yolov7_640x640_ppu.dxnn -v assets/videos/drone-city-road2.mov {"" if SHOW_WINDOW else "--no-display"}

### 2.3 Pose Estimation
- **bin/yolov8-s-pose_640x640_sync** runs a pose estimation model that detects people and estimates their body keypoints in images, videos, or a camera.

In [ ]:
%cd {DX_APP_DIR}

# Image input, headless
!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -i sample/img/sample_person_b.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# On a desktop session (opens a window):
#!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -v assets/videos/dance-solo.mov
#!./bin/yolov8-s-pose_640x640_sync -m assets/models/yolov8-s-pose_640x640.dxnn -c 0

### 2.4 Segmentation
- **bin/yolov8-n-seg_640x640_async** runs an instance segmentation model: a box and a pixel mask for every object it finds in images, videos, or a camera.

In [ ]:
%cd {DX_APP_DIR}

# Image input, headless
!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -i sample/img/sample_person_b.jpg --no-display --save --save-dir "{DEMO_OUTPUT_DIR}"
show_latest_image(DEMO_OUTPUT_DIR)

# On a desktop session (opens a window):
#!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -v assets/videos/blackbox-city-road.mp4
#!./bin/yolov8-n-seg_640x640_async -m assets/models/yolov8-n-seg_640x640.dxnn -c 0

### 2.5 Demo launcher: run_demo.sh

`run_demo.sh` launches **26 demos**, one per task, in up to **6 execution modes**: C++ sync and async, and the four Python variants. It asks three questions (demo, mode, input), so in a terminal you start it and answer the menus (**File > New > Terminal**):

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./run_demo.sh
```

The next cell prints the same commands with your actual path. Before the first menu, the launcher downloads any of the 26 demo models that are missing, so the first start can take a few minutes. It also finds the `venv-dx-runtime` Python by itself, so the Python modes need no activated virtual environment.

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./run_demo.sh")


The first menu lists the demos:

| # | Demo | # | Demo |
|---|---|---|---|
| 0 | Object Detection (YOLOv7) | 13 | Image Enhancement (Zero-DCE) |
| 1 | Object Detection (YOLOv11N) | 14 | Embedding (ArcFace) |
| 2 | Face Detection (SCRFD500M) | 15 | Attribute Recognition (DeepMAR-ResNet50) |
| 3 | OBB Detection (YOLO26N-OBB) | 16 | PPU Pipeline (YOLOv7-PPU) |
| 4 | Pose Estimation (YOLOv8s-Pose) | 17 | Keypoint Detection (SuperPoint) |
| 5 | Hand Landmark (HandLandmarkLite) | 18 | Object Pose Estimation (DOPE) |
| 6 | Face Alignment (3DDFA-V2-MobileNetV1) | 19 | Panoptic Driving (YOLOPv2) |
| 7 | Instance Segmentation (YOLOv8N-Seg) | 20 | 3D Object Detection (SFA3D) |
| 8 | Semantic Segmentation (DeepLabV3+MobileNet) | 21 | Hand Detection (MediaPipe Palm) |
| 9 | Classification (ResNet50) | 22 | Image Retrieval (CLIP RN50) |
| 10 | Depth Estimation (YOLO26-Depth-S) | 23 | Visual Place Recognition (EigenPlaces R18) |
| 11 | Image Denoising (DnCNN-50) | 24 | Person Re-ID (RepVGG-A0) |
| 12 | Super Resolution (ESPCN-X4) | 25 | Image Matting (PP-Matting HRNet-W48) |

The second menu picks the execution mode. Modes 1 and 2 run the C++ executables, and modes 3 to 6 run the four Python variants that section 3.4 explains:

> ![](assets/sc-run_demo2.png)

The third menu picks the input: `1` for the demo's sample video, `2` for its sample image.

The three answers can also be given as options, which skips every menu: `--task`, `--mode`, and `--input`. The next cell prints the launcher's help. When `RUN_DEMO_LAUNCHER = True`, it then starts demo `0` (YOLOv7) in mode `2` (C++ async) on the sample image. The demo opens a window, so set the flag on a desktop session only.

In [ ]:
RUN_DEMO_LAUNCHER = False   # True starts the demo below; it opens a window (desktop session only)
%cd {DX_APP_DIR}

!./run_demo.sh --help

if RUN_DEMO_LAUNCHER:
    # Demo 0 (Object Detection, YOLOv7), mode 2 (cpp_async), input 2 (image): no menus
    !./run_demo.sh --task 0 --mode 2 --input 2

### 2.6 Find, inspect, and export examples with dx_tool.sh

`scripts/dx_tool.sh` manages the examples. Started without arguments, it opens an interactive menu, but its query commands also run directly, which makes them handy in a notebook:

| Command | What it does |
|---|---|
| `dx_tool.sh list [FILTER]` | List the examples, optionally filtered by task (for example `det`) |
| `dx_tool.sh search <KEYWORD>` | Find examples by model name or task |
| `dx_tool.sh info <MODEL>` | Show one example's files, factory classes, and `config.json` |
| `dx_tool.sh run` / `dx_tool.sh bench` | Run or benchmark many examples at once (interactive filter) |
| `dx_tool.sh extract` | Export a standalone package (interactive) |

The next cell searches for the YOLO26-S examples and prints the details of the detector. `search` prints every hit as `<task>/<family>/<variant>`, which is exactly the form that the export script in section 3 takes. `info` shows, among other things, the default sample image and video and the score and NMS thresholds in `config.json`.

In [ ]:
%cd {DX_APP_DIR}

# Which examples exist for YOLO26-S? (C++ and Python, every task)
!./scripts/dx_tool.sh search yolo26-s

# Details of one variant: files, factory classes, and config.json
!./scripts/dx_tool.sh info yolo26-s_640x640

The interactive menu adds the export. It asks questions, so run it in a terminal (File > New > Terminal):

```bash
cd <DX_ALL_SUITE_DIR>/dx-runtime/dx_app
./scripts/dx_tool.sh
```

The next cell prints the same commands with your actual path. When the menu appears, follow the steps below it.

In [ ]:
# Interactive script: run these two commands in a terminal (File > New > Terminal).
print(f"cd {shlex.quote(str(DX_APP_DIR))}")
print("./scripts/dx_tool.sh")


When the menu appears:
- Choose `2` (Extract Package).
- Select the programming language: `1` C++, `2` Python, or `3` both.
- Type the example path in the `<task>/<family>/<variant>` form, for example `object_detection/yolo26/yolo26-s_640x640`. Tab completes the path.
- Press Enter to accept the default output directory `output`.

> ![](assets/sc-extract.png)

The screenshot comes from an earlier release, where the path had only two levels (`object_detection/yolo26s`). A few seconds later, the packages are written to `output/cpp/<task>/<family>/<variant>` and `output/py/<task>/<family>/<variant>`.

The menu calls `scripts/extract_model_package.sh` behind the scenes. Section 3 uses that script directly, because it takes the model and language as arguments and therefore runs inside a notebook cell.

## 3. Run the extracted standalone examples

Section 2.6 showed the interactive export. This section does the same thing from the notebook with `scripts/extract_model_package.sh`, then builds and runs the exported packages **outside the DX-APP tree**, exactly as you would in your own project:

| Section | Example (`<task>/<family>/<variant>`) | Language and variant | What it shows |
|---|---|---|---|
| 3.2 | `object_detection/yolo26/yolo26-s_640x640` | Python, `*_sync.py` | The simplest way to run any Model Zoo model |
| 3.3 | `object_detection/yolo26/yolo26-s_640x640` | C++, `*_sync` | Building the exported C++ package with CMake |
| 3.4 | `pose_estimation/yolo26_pose/yolo26-s-pose_640x640` | Python, `*_async_cpp_postprocess.py` | Another task, with C++ post-processing |
| 3.5 | `object_detection/yolo26/yolo26-s_640x640` | Python, all four variants | Measuring which variant to use |

Every exported Python example imports `dx_engine` and `dx_postprocess`, which DX-Runtime installed into `dx-runtime/venv-dx-runtime`. The cells therefore call that interpreter explicitly, and they export the Python packages with `--no-prune` (section 3.1 explains why). Inside the DX-APP tree an example falls back to a default sample image when you give no input, but an exported package contains no sample images. The cells therefore pass an image from `dx_app/sample/img`, run headless (`--no-display --save`), and show the saved result in the notebook.

### 3.1 Export the packages

Start with the script's own help. It lists the arguments the next cells use: the example as `<task>/<family>/<variant>` (or `<task>/<family>` for every variant of a family), `--lang cpp|py|both`, `--output-dir`, and `--no-prune`.

In [ ]:
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh --help


Without `--output-dir` the script prepares the package in place inside `dx_app`; with it, the package is copied to `<dir>/cpp/<task>/<family>/<variant>` and `<dir>/py/<task>/<family>/<variant>`. The next cell downloads the two YOLO26 models and exports three packages into this tutorial's `workspace/` directory, which git ignores: the C++ object detector, and the Python object detector and pose estimator.

In [ ]:
STANDALONE_DIR = WORKSPACE_DIR / "standalone"
OUTPUT_DIR = WORKSPACE_DIR / "standalone-outputs"
DX_RUNTIME_PYTHON = DX_RUNTIME_DIR / "venv-dx-runtime" / "bin" / "python"   # has dx_engine and dx_postprocess
SAMPLE_IMG_DIR = DX_APP_DIR / "sample" / "img"
MODEL_DIR = DX_WORKSPACE_DIR / "res" / "models"

for path in (STANDALONE_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)
if not DX_RUNTIME_PYTHON.is_file():
    raise FileNotFoundError(f"DX-Runtime Python environment not found: {DX_RUNTIME_PYTHON} (Tutorial 01, section 3)")

# Both models were downloaded in Tutorial 01 section 3.3; this only downloads what is missing.
!cd "{DX_APP_DIR}" && bash setup.sh --models yolo26-s_640x640 yolo26-s-pose_640x640 --no-force

# Export: object detection in C++ (pruned, the default) and in Python, pose estimation in Python.
# The Python packages use --no-prune; the next markdown cell explains why.
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang cpp --output-dir "{STANDALONE_DIR}"
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang py --output-dir "{STANDALONE_DIR}" --no-prune
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh pose_estimation/yolo26_pose/yolo26-s-pose_640x640 --lang py --output-dir "{STANDALONE_DIR}" --no-prune

!tree -L 5 "{STANDALONE_DIR}"

#### Pruned packages and `--no-prune`

By default the export **prunes** `common/`: it copies only the shared files that this example actually uses, instead of the whole runtime layer. The package is smaller and easier to read, but it can run only this example. `--no-prune` copies the full `common/`, which you also need if you plan to add a second model to the same package. The C++ package from the previous cell is pruned. The next cell exports the Python YOLO26-S example once more with the default pruning and compares it with the full copy.

> **Note for DX-APP v3.3.0:** a pruned **Python** package misses processor classes that its `config.json` names, so it stops at start-up with `VariantConfigError: processor class 'LetterboxPreprocessor' is exported by neither common.processors nor common.visualizers`. Until this is fixed, export Python packages with `--no-prune`, as this tutorial does. Pruned C++ packages are not affected.

In [ ]:
PRUNED_DIR = WORKSPACE_DIR / "standalone-pruned"
!cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh object_detection/yolo26/yolo26-s_640x640 --lang py --output-dir "{PRUNED_DIR}"


def count_files(folder):
    return sum(1 for p in Path(folder).rglob("*") if p.is_file())


PACKAGE = Path("py") / "object_detection" / "yolo26" / "yolo26-s_640x640"
print(f"\n{'package':<14}{'all files':>10}{'common/ files':>15}")
for label, root in (("pruned", PRUNED_DIR), ("--no-prune", STANDALONE_DIR)):
    folder = root / PACKAGE
    print(f"{label:<14}{count_files(folder):>10}{count_files(folder / 'common'):>15}")

#### The model card

Every example folder carries a **model card** generated from the Model Zoo. `README.md` describes the model: what it is, its accuracy and speed with their source, its downloads, and how to run it. Python example folders also have `model.html`, the same card for offline viewing in a browser, and `result.webp`, a reference result. The exported Python package keeps all three. The next cell shows the reference image and the card of the package you just exported.

In [ ]:
import re

CARD_DIR = STANDALONE_DIR / "py" / "object_detection" / "yolo26" / "yolo26-s_640x640"
!ls "{CARD_DIR}"

display(Image(filename=str(CARD_DIR / "result.webp"), width=480))   # the reference result
card = (CARD_DIR / "README.md").read_text()
display(Markdown(re.sub(r"!\[[^\]]*\]\(result\.webp\)", "", card)))  # the card; its image is shown above
print("Offline HTML card:", CARD_DIR / "model.html")

### 3.2 Example 1: Python standalone, object detection

`yolo26-s_640x640_sync.py` is the plain synchronous variant: pure-Python post-processing, one frame at a time. It is the variant to read first. The arguments are the same for every Python example in DX-APP: `--model`, one input option (`--image`, `--video`, `--camera`, `--rtsp`), and `--no-display --save` for headless runs.

```bash
cd <workspace>/standalone/py/object_detection/yolo26/yolo26-s_640x640
<dx-runtime>/venv-dx-runtime/bin/python yolo26-s_640x640_sync.py \
    --model <workspace>/res/models/yolo26-s_640x640.dxnn \
    --image <dx_app>/sample/img/sample_street.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
EXAMPLE_DIR = STANDALONE_DIR / "py" / "object_detection" / "yolo26" / "yolo26-s_640x640"
MODEL = MODEL_DIR / "yolo26-s_640x640.dxnn"
IMAGE = SAMPLE_IMG_DIR / "sample_street.jpg"

!cd "{EXAMPLE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26-s_640x640_sync.py --model "{MODEL}" --image "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.3 Example 2: C++ standalone, object detection

The exported C++ package contains `yolo26-s_640x640_sync.cpp`, `yolo26-s_640x640_async.cpp`, the shared `common/` layer, and a generated `CMakeLists.txt` that links against the installed DX-RT. Building it is the standard CMake sequence; nothing refers back to the DX-APP tree.

```bash
cd <workspace>/standalone/cpp/object_detection/yolo26/yolo26-s_640x640
mkdir -p build && cd build
cmake .. && make -j$(nproc)
./yolo26-s_640x640_sync -m <workspace>/res/models/yolo26-s_640x640.dxnn \
    -i <dx_app>/sample/img/sample_street.jpg --no-display --save --save-dir <output-dir>
```

Note that the C++ examples spell the options `-m/--model_path` and `-i/--image_path`; `./yolo26-s_640x640_sync --help` lists them. The build takes one or two minutes.

In [ ]:
CPP_DIR = STANDALONE_DIR / "cpp" / "object_detection" / "yolo26" / "yolo26-s_640x640"

!cd "{CPP_DIR}" && mkdir -p build && cd build && cmake .. && make -j$(nproc)

!cd "{CPP_DIR}/build" && ./yolo26-s_640x640_sync -m "{MODEL}" -i "{IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.4 Example 3: Python standalone, pose estimation, async with C++ post-processing

Every Python example comes in four variants. The file name tells you which one you are running:

| Variant | Post-processing | Threading | Use it for |
|---|---|---|---|
| `*_sync.py` | Pure Python | Synchronous | Learning and debugging the logic |
| `*_async.py` | Pure Python | Asynchronous | Higher throughput; the usual starting point for deployment |
| `*_sync_cpp_postprocess.py` | C++ binding (`dx_postprocess`) | Synchronous | Heavy post-processing, one frame at a time |
| `*_async_cpp_postprocess.py` | C++ binding (`dx_postprocess`) | Asynchronous | Heavy post-processing at a high frame rate |

> In DX-APP v3.2.x the `*_cpp_postprocess.py` scripts silently ran the Python post-processor; v3.3.0 fixed this. Measurements taken with an older release therefore show no difference between the variants.

This example switches to a different task and to the asynchronous C++ post-processing variant. The command shape does not change: only the directory, the script name, and the model do.

```bash
cd <workspace>/standalone/py/pose_estimation/yolo26_pose/yolo26-s-pose_640x640
<dx-runtime>/venv-dx-runtime/bin/python yolo26-s-pose_640x640_async_cpp_postprocess.py \
    --model <workspace>/res/models/yolo26-s-pose_640x640.dxnn \
    --image <dx_app>/sample/img/sample_people.jpg \
    --no-display --save --save-dir <output-dir>
```

In [ ]:
POSE_DIR = STANDALONE_DIR / "py" / "pose_estimation" / "yolo26_pose" / "yolo26-s-pose_640x640"
POSE_MODEL = MODEL_DIR / "yolo26-s-pose_640x640.dxnn"
POSE_IMAGE = SAMPLE_IMG_DIR / "sample_people.jpg"

!cd "{POSE_DIR}" && "{DX_RUNTIME_PYTHON}" yolo26-s-pose_640x640_async_cpp_postprocess.py --model "{POSE_MODEL}" --image "{POSE_IMAGE}" --no-display --save --save-dir "{OUTPUT_DIR}"

show_latest_image(OUTPUT_DIR)

### 3.5 Measure before you choose a variant

Which of the four variants is fastest depends on where the time goes. When the environment variable `DXAPP_PROGRESS=1` is set, every example prints a `[PROGRESS]` line about once per second and a final one at the end. Each line holds the throughput (`fps`, `avg_fps`) and the average time per frame of every stage: reading the input (`read_ms`), pre-processing (`pre_ms`), NPU inference (`infer_ms`), and post-processing (`post_ms`).

The next cell runs all four YOLO26-S variants headless on the same 588-frame video and tabulates the final line of each. One run looks like this:

```bash
DXAPP_PROGRESS=1 <dx-runtime>/venv-dx-runtime/bin/python yolo26-s_640x640_sync.py \
    --model <workspace>/res/models/yolo26-s_640x640.dxnn --video <dx_app>/assets/videos/boat.mp4 --no-display
```

The cell takes about 40 seconds.

In [ ]:
VIDEO = DX_APP_DIR / "assets" / "videos" / "boat.mp4"
VARIANT_SCRIPTS = [
    "yolo26-s_640x640_sync.py",
    "yolo26-s_640x640_async.py",
    "yolo26-s_640x640_sync_cpp_postprocess.py",
    "yolo26-s_640x640_async_cpp_postprocess.py",
]

rows = []
for script in VARIANT_SCRIPTS:
    log = !cd "{EXAMPLE_DIR}" && DXAPP_PROGRESS=1 "{DX_RUNTIME_PYTHON}" {script} --model "{MODEL}" --video "{VIDEO}" --no-display 2>&1
    progress = [line for line in log if line.startswith("[PROGRESS]")]
    if not progress:
        print("\n".join(log[-20:]))
        raise RuntimeError(f"{script} printed no [PROGRESS] line; see its output above")
    rows.append((script, dict(re.findall(r"(\w+)=([\d.]+)", progress[-1]))))

print(f"{'variant':<44}{'avg_fps':>9}{'pre_ms':>8}{'infer_ms':>10}{'post_ms':>9}")
for script, m in rows:
    print(f"{script:<44}{m['avg_fps']:>9}{m['pre_ms']:>8}{m['infer_ms']:>10}{m['post_ms']:>9}")

How to read the table (your numbers depend on the host CPU and the NPU card):

- **Async against sync:** the asynchronous variants keep several frames in flight on the NPU, so they reach a much higher frame rate. On one DX-M1 card YOLO26-S runs at roughly 200 FPS asynchronously against roughly 50 FPS synchronously. `infer_ms` grows in async mode because it measures the turnaround of each frame from submission to result, not the NPU time alone.
- **C++ against Python post-processing:** the `cpp_postprocess` variants cut `post_ms`, but for YOLO26-S post-processing takes well under a millisecond, so the frame rate barely changes. C++ post-processing pays off when `post_ms` is a large share of the frame time, for example with segmentation masks, many anchors, or a slow host CPU.
- **Rule of thumb:** start with `*_async.py`, measure with `DXAPP_PROGRESS=1`, and switch to a `cpp_postprocess` variant only when post-processing is the bottleneck.

### 3.6 Try it: any model from the Model Zoo

The examples followed one pattern, and it holds for all 497 variants:

1. Pick a model on the [Model Zoo](https://developer.deepx.ai/modelzoo/) page and find its example with `./scripts/dx_tool.sh search <keyword>`.
2. Run it in place. Inside the DX-APP tree the model and the input are optional: without `--model` an example loads `assets/models/<variant>.dxnn` and downloads it first if it is missing, and without an input it uses its task's default sample image.
3. Export it when you want to build on it: `scripts/extract_model_package.sh <task>/<family>/<variant> --lang py --output-dir <dir>`.

The next cell looks up a variant in `config/model_registry.json` and runs its synchronous Python example in place with no model and no input, the shortest possible run. It shows the result and prints the export command. Change `VARIANT` to try another model, for example `resnet50_224x224` (classification), `yolov12-n_640x640` (YOLOv12), or `rtdetr-r18vd-6x_640x640` (RT-DETR). A model that is not on disk yet is downloaded first, which takes a few seconds to a minute.

In [ ]:
import json

VARIANT = "scrfd-500m_640x640"   # change me: any variant (the .dxnn file stem) from config/model_registry.json

registry = json.loads((DX_APP_DIR / "config" / "model_registry.json").read_text())
entry = next((e for e in registry if VARIANT in (e["variant"], e["model_name"])), None)
if entry is None:
    raise KeyError(f"{VARIANT} is not in the registry. Search with: cd {DX_APP_DIR} && ./scripts/dx_tool.sh search {VARIANT}")

task, family, variant = entry["task"], entry["family"], entry["variant"]
SCRIPT = f"src/python_example/{task}/{family}/{variant}/{variant}_sync.py"
print(f"{variant}: task={task}, family={family}, model file={entry['dxnn_file']}\n")

# The shortest run: no --model and no input, so the example uses its default model and sample image.
!cd "{DX_APP_DIR}" && "{DX_RUNTIME_PYTHON}" {SCRIPT} --no-display --save --save-dir "{OUTPUT_DIR}"
show_latest_image(OUTPUT_DIR)

print("\nExport it as a standalone package:")
print(f'cd "{DX_APP_DIR}" && bash scripts/extract_model_package.sh {task}/{family}/{variant} --lang py --output-dir "{STANDALONE_DIR}"')

## 4. Multi-model graph: chain models without writing code

Real applications often run more than one model per frame: a detector finds people and a second model reads their attributes, or a palm detector finds hands and a landmark model places the finger joints. DX-APP v3.3.0 adds the **multi-model graph** for this. You describe the pipeline in a JSON file, and one executable runs it:

- **nodes** are the input source and the models, each model named by its variant;
- **edges** connect them; an edge with `"roi"` crops every box that the first model found and feeds the crops to the next model.

There is no C++ to edit and nothing to rebuild: to try a different model, you change one string in the JSON file. The executables are `bin/multi_model_graph_sync` and `bin/multi_model_graph_async`, and the sample graphs are in `src/cpp_example/multi_model_graph/`.

| Option | Effect |
|---|---|
| `--graph <file>` | Run the graph |
| `--check <file>` | Validate the graph and its model files without using the NPU |
| `--output <file>` | Save the composited result image |
| `--report <file>` | Save the results of every node as JSON |
| `--list-models` | List the models a graph can use (`--consumes roi` lists second-stage models) |

> The Python module `dx_graph` runs the same graphs from Python. It is built only by `./build.sh --all`, so this tutorial uses the executables.

### 4.1 Build the graph executables and download the models

The next cell builds the two executables if your DX-APP build does not contain them yet, which takes about one minute. It then downloads the three extra models that the graphs in this section use: a palm detector, a hand landmark model, and a person-attribute model.

In [ ]:
%cd {DX_APP_DIR}
GRAPH_DIR = DX_APP_DIR / "src" / "cpp_example" / "multi_model_graph"
GRAPH_OUTPUT_DIR = WORKSPACE_DIR / "graph-outputs"
GRAPH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not (DX_APP_DIR / "bin" / "multi_model_graph_async").is_file():
    !./build.sh --target multi_model_graph_sync multi_model_graph_async
!ls bin/multi_model_graph_*

!bash setup.sh --models mediapipe-hand-detector_192x192 mediapipe-hands-lite_224x224 deepmar_resnet50_224x224 --no-force

### 4.2 A two-stage cascade: palm detection, then hand landmarks

`hand_cascade.json` is the smallest sample graph:

In [ ]:
!cat "{GRAPH_DIR}/hand_cascade.json"

The source `cam` is a sample image. `palm` finds hands in the full frame. The edge from `palm` to `landmark` has `"roi"`, so every hand box, padded by 15%, is cropped and passed to the landmark model.

`--check` validates the graph before anything runs on the NPU: it resolves each model, shows what it `produces` and `consumes`, and confirms that the `.dxnn` file is present. `--graph` then runs it, `--output` saves the composited image, and `--report` writes the results of every node. The sample graphs use paths relative to `dx_app`, so run them from there.

In [ ]:
%cd {DX_APP_DIR}

!./bin/multi_model_graph_async --check src/cpp_example/multi_model_graph/hand_cascade.json

!./bin/multi_model_graph_async --graph src/cpp_example/multi_model_graph/hand_cascade.json --output "{GRAPH_OUTPUT_DIR}/hand_cascade.jpg" --report "{GRAPH_OUTPUT_DIR}/hand_cascade.json"

display(Image(filename=str(GRAPH_OUTPUT_DIR / "hand_cascade.jpg"), width=480))

The report lists, for every frame, the results of each node. Models that ran on the full frame are under `nodes`; models that ran on crops are under `roi_nodes`, together with the box in the frame that each crop came from. The next cell summarizes the first frame.

In [ ]:
report = json.loads((GRAPH_OUTPUT_DIR / "hand_cascade.json").read_text())
frame = report["frames"][0]
for name, node in frame["nodes"].items():
    payload = node["payload"]
    print(f"{name:<10} full frame  {payload['shape']:<10} items={len(payload.get('items', []))}")
for name, crops in frame.get("roi_nodes", {}).items():
    for crop in crops:
        box = [round(v) for v in crop["origin"]["src_box"]]
        print(f"{name:<10} crop        {crop['payload']['shape']:<10} from box {box}")

### 4.3 Swap a model by editing the JSON

`cascade_od_attr.json` detects people and passes every person crop to `deepmar_resnet50_224x224`, a person-attribute model (gender, age group, clothing, and so on). Its detector is `yolov8-n_640x640`, which this tutorial has not downloaded. Instead of downloading it, the next cell writes a copy of the graph into the workspace with the detector replaced by `yolo26-s_640x640`, the model from section 3. Because the copy lives outside `dx_app`, the cell also makes the source image path absolute. It then checks and runs the edited graph; nothing is rebuilt.

In [ ]:
graph = json.loads((GRAPH_DIR / "cascade_od_attr.json").read_text())
for node in graph["nodes"]:
    if node.get("type") == "source":
        node["uri"] = str(DX_APP_DIR / node["uri"])   # absolute path: the copy is outside dx_app
    if node["id"] == "od":
        node["model"] = "yolo26-s_640x640"           # the one-string model swap

MY_GRAPH = GRAPH_OUTPUT_DIR / "person_attributes_yolo26.json"
MY_GRAPH.write_text(json.dumps(graph, indent=2) + "\n")
!cat "{MY_GRAPH}"

!./bin/multi_model_graph_async --check "{MY_GRAPH}"
!./bin/multi_model_graph_async --graph "{MY_GRAPH}" --output "{GRAPH_OUTPUT_DIR}/person_attributes.jpg"

display(Image(filename=str(GRAPH_OUTPUT_DIR / "person_attributes.jpg"), width=640))

### 4.4 Which models fit, and what else there is

Not every model fits every slot. A second-stage model must accept a crop: its `consumes` column must be `roi` or `either`. `--list-models --consumes roi` filters the registry for these models; the `file` column shows whether the `.dxnn` is on disk. The next cell prints the first lines and the total count.

`run_multi_model_demo.sh --list` then lists the sample graphs. They cover cascades (a detector feeding attribute, re-identification, or classification models), fan-out (detection, segmentation, and pose on the same frame), chains (super resolution or low-light enhancement before detection), and logic nodes that filter, count, track, and check zones or lines. `bash run_multi_model_demo.sh --check` validates every sample graph without using the NPU and prints the download command for each missing model.

In [ ]:
%cd {DX_APP_DIR}

# Second-stage candidates: the first lines, then the number of models
!./bin/multi_model_graph_async --list-models --consumes roi | head -12
!./bin/multi_model_graph_async --list-models --consumes roi | tail -n +3 | wc -l

# The sample graphs and the demo scenarios
!bash run_multi_model_demo.sh --list

# Validate every sample graph (no NPU use; long output):
#!bash run_multi_model_demo.sh --check

## 5. Troubleshooting

| Symptom | What you see | Likely cause | Fix |
|---|---|---|---|
| An old executable name is not found | `./bin/yolov7_async: No such file or directory` | DX-APP v3.3.0 names executables after the variant | Use the new name, such as `./bin/yolov7_640x640_async`; `ls bin` lists them |
| An executable is missing | `./bin/<variant>_sync: No such file or directory` | The default build (`--minimal`) builds only the 26 demo models | Run `./build.sh --target <variant>` in `dx_app`, or `./build.sh --all`, in a terminal |
| A model is refused at load time | `.dxnn container v9 needs DX-RT >= 3.5.0, but this runtime is 3.4.1` or `Model file format version 9 is not supported` | Model Zoo `2_5_0` files need DX-RT 3.5.0 or later | Update DX-Runtime (Tutorial 01 section 3); `dxrt-cli --version` shows the installed version |
| `import dx_engine` fails | `dx_engine and DX-RT versions do not match` | `venv-dx-runtime` holds a `dx_engine` built for another DX-RT | Reinstall DX-Runtime so that both are 3.5.x |
| A model cannot be loaded | `Failed to load model` with a path under `assets/models` | The model was not downloaded | Run the download cell in section 1 with the variant name, then rerun |
| A graph check reports a missing model | `MISSING: 1 model file is not in .../assets/models` | The graph names a model that was not downloaded | Run the download command that `--check` prints, or `bash setup.sh --models <variant> --no-force` |
| No output window | `cannot open display` or the cell ends without a window | No graphical session (SSH or headless host) | Run on a desktop session, or keep `--no-display --save` and look at the saved result |
| The camera cannot be opened | `Could not open camera 0` | No `/dev/video0`, or the user is not in the `video` group | `v4l2-ctl --list-devices`; `sudo usermod -aG video $USER` and log in again |
| `run_demo.sh` stops before the menu | `Failed to download demo models.` | No network access during the automatic demo-model download | Connect, or run `./setup.sh --demo-models` once with network access |
| An exported Python example stops at start-up | `VariantConfigError: processor class '...' is exported by neither common.processors nor common.visualizers` | The package was exported with the default pruning, which misses classes named in `config.json` (DX-APP v3.3.0) | Export it again with `--no-prune` |
| An exported Python example cannot import `dx_engine` | `ModuleNotFoundError: No module named 'dx_engine'` | It was run with the Jupyter Python instead of the DX-Runtime environment | Use `<dx-runtime>/venv-dx-runtime/bin/python`, as the cells in section 3 do |
| The exported C++ package does not configure | `Could not find a package configuration file provided by "dxrt"` | DX-RT is not installed, or it is under a custom prefix | Install DX-Runtime (Tutorial 01 section 3) or add `-DCMAKE_PREFIX_PATH=<prefix>` to the `cmake` line |
| An exported example finds no input | `Input file not found: sample/img/...` | The package contains no sample images | Pass `--image` (Python) or `-i` (C++) with a file from `dx_app/sample/img` |

## 6. Summary

### 6.1 DX-APP workflow completed

```text
Load SDK paths
       │
       ▼
Download the selected models by variant name
       │
       ▼
Run C++ or Python examples headless and view the saved results
       │
       ├── Synchronous
       ├── Asynchronous
       └── PPU
       │
       ▼
Find an example with dx_tool.sh, export it, and measure its variants
       │
       ▼
Chain several models with a multi-model graph
```

### 6.2 Learning dashboard

| Area | What you can now do |
|---|---|
| SDK paths | Load `DX_APP_DIR` and related paths from `config.json` |
| Naming | Find an example, its executables, and its scripts from the variant name in `<task>/<family>/<variant>` |
| Model setup | Download only the models and videos required for an exercise, by variant name |
| Execution modes | Choose synchronous, asynchronous, or PPU execution for the intended goal |
| Measurement | Compare the four Python variants with `DXAPP_PROGRESS=1` before choosing one |
| Demo launcher | Start a demo with `run_demo.sh`, from the menus or with `--task`, `--mode`, and `--input` |
| Standalone code | Export any model's example with `scripts/extract_model_package.sh`, read its model card, then build and run it outside DX-APP |
| Model Zoo coverage | Find the DX-APP example for any Model Zoo model through `config/model_registry.json` or `dx_tool.sh search` |
| Multi-model graph | Check, run, and edit a graph JSON that chains several models |

### 6.3 Completion checklist

- [ ] Confirm that the DEEPX NPU is visible to DX-RT and that DX-RT is 3.5.0 or later
- [ ] Prepare the required DX-APP model and video assets
- [ ] Run at least one end-to-end inference example and view its saved result
- [ ] Compare the available execution modes
- [ ] Locate a reference implementation for a target task with `dx_tool.sh search`
- [ ] Open `run_demo.sh`
- [ ] Export, build, and run the standalone Python and C++ examples outside the DX-APP tree
- [ ] Measure the four Python variants and pick one for your model
- [ ] Run a Model Zoo model that you chose yourself
- [ ] Run a multi-model graph and swap one of its models

> **Next:** Continue to Tutorial 03 to adapt a trained model, compile it to DXNN, and integrate it with a DX-APP runtime configuration.